# 01: Load Excel reports into Bronze

Both sample reports contain some of the same record IDs. Bronze keeps all source rows and records their filenames and ingestion times. Each run appends the rows again, including files loaded on earlier runs.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from pyspark.sql import functions as F

INPUT_DIR = Path('/lakehouse/default/Files/sample_reports')
SOURCE_FILES = ['january_mtd.xlsx', 'january_ytd.xlsx']
BRONZE_TABLE = 'demo_bronze_activity'

frames = []
for filename in SOURCE_FILES:
    file_path = INPUT_DIR / filename
    if not file_path.exists():
        raise FileNotFoundError(f'Upload {filename} to {INPUT_DIR} before running this notebook')
    # Pandas handles the Excel format; Spark handles the lakehouse writes.
    frame = pd.read_excel(file_path, dtype={'record_id': str})
    frame.columns = [str(col).strip().lower() for col in frame.columns]
    frame['source_file'] = filename
    frames.append(frame)

raw = pd.concat(frames, ignore_index=True)
if raw.empty:
    raise ValueError('No report rows found')

spark_df = spark.createDataFrame(raw.astype(str))
spark_df = spark_df.withColumn('ingested_at', F.current_timestamp())
spark_df.write.format('delta').mode('append').saveAsTable(BRONZE_TABLE)
print(f'Appended {spark_df.count()} rows to {BRONZE_TABLE}')
